# **2. DATA UNDERSTANDING**

## **LIBRARIES**

In [ ]:
! pip install adlfs azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.9/412.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.9/187.9 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.9/116.9 kB 10.1 MB/s eta 0:00:00


In [ ]:
! pip install folium

In [ ]:
# Standard Library
import os
import io

# DATA MANIPULATION AND ANALYSIS
import pandas as pd
import numpy as np

# DATA VISUALISATION
import matplotlib.pyplot as plt
import seaborn as sns

# AZURE CLOUD STORAGE
from adlfs import AzureBlobFileSystem
from azure.storage.blob import BlobServiceClient
from io import BytesIO

# Geographical map
import folium

# Regular expressions
import re

In [ ]:
# Correctly set pandas display option for max columns and rows
pd.options.display.max_columns = None
pd.options.display.max_rows = None

## **EXTRACT FROM AZURE BLOB STORAGE**

In [ ]:
# Configuration
AZURE_STORAGE_ACCOUNT = os.environ.get("AZURE_STORAGE_ACCOUNT", "researchprojectx24104515")
AZURE_STORAGE_KEY = os.environ.get("AZURE_STORAGE_KEY", "bxpexO6i+Hz6n1WiipTn+sTCuLPGMS1BogMERrIrHd16DpQ0GLfQ0R33yrSw4MxsDomq5yNMgw1o+AStlx/MjA==")
CONTAINER_NAME = "counter-locations"

# Establish connection
connection_string = f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

fs = AzureBlobFileSystem(
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)

# Load CSV from Blob
def load_csv(container_client, blob_name):
    blob_client = container_client.get_blob_client(blob_name)
    with BytesIO() as input_blob:
        blob_client.download_blob().download_to_stream(input_blob)
        input_blob.seek(0)
        df = pd.read_csv(input_blob)
    return df

# Load file
counters = load_csv(container_client, "counter_locations.csv")

## **EXPLORATORY DATA ANALYSIS**

In [ ]:
counters.describe()

,_id,Latitude,Longitude
count,131.000000,86.000000,86.000000
mean,66.000000,53.333783,-5.562965
std,37.960506,0.031422,6.427245
min,1.000000,53.250970,-6.422330
25%,33.500000,53.315535,-6.264362
50%,66.000000,53.344060,-6.258875
75%,98.500000,53.348782,-6.240983
max,131.000000,53.409854,53.344410


In [ ]:
counters.head()

,_id,Bike Counter Locations,Latitude,Longitude,Setup Date,User Type
0,1,Aston Quay/Fitzgeralds,53.34668,-6.25988,2019-02-27T00:00:00,Pedestrians
1,2,Bachelors walk/Bachelors way,53.34744,-6.26010,2019-02-27T00:00:00,Pedestrians
2,3,Baggot st lower/Wilton tce inbound,53.33454,-6.24562,2020-08-27T00:00:00,Pedestrians
3,4,Baggot st upper/Mespil rd/Bank,53.33377,-6.24451,2020-08-27T00:00:00,Pedestrians
4,5,Capel st/Mary street,53.34853,-6.26880,2011-08-01T00:00:00,Pedestrians


In [ ]:
counters = counters.dropna(subset=["Latitude", "Longitude", "User Type"])
counters = counters[["Bike Counter Locations", "Latitude", "Longitude", "User Type"]]
counters

,Bike Counter Locations,Latitude,Longitude,User Type
0,Aston Quay/Fitzgeralds,53.346680,-6.259880,Pedestrians
1,Bachelors walk/Bachelors way,53.347440,-6.260100,Pedestrians
2,Baggot st lower/Wilton tce inbound,53.334540,-6.245620,Pedestrians
3,Baggot st upper/Mespil rd/Bank,53.333770,-6.244510,Pedestrians
4,Capel st/Mary street,53.348530,-6.268800,Pedestrians
5,Castleknock Totem,53.370000,-6.349436,Cyclists
6,Charleville Mall,53.356950,-6.245480,Cyclists
7,Clonskeagh Totem,53.306376,-6.237057,Cyclists
8,Clontarf - James Larkin Rd,53.368540,-6.170320,Cyclists
9,Clontarf - Pebble Beach Carpark,53.358250,-6.191070,Cyclists


In [ ]:
def normalize_name(name):
    name = name.lower()                       # Lower case
    name = re.sub(r"[^a-z0-9 ]", " ", name)  # Replace any character to space
    name = re.sub(r"\s+", " ", name)         # Replace multiple space to only one
    return name.strip()

counters["Bike Counter Locations"] =  counters["Bike Counter Locations"].apply(normalize_name)
counters

,Bike Counter Locations,Latitude,Longitude,User Type
0,aston quay fitzgeralds,53.346680,-6.259880,Pedestrians
1,bachelors walk bachelors way,53.347440,-6.260100,Pedestrians
2,baggot st lower wilton tce inbound,53.334540,-6.245620,Pedestrians
3,baggot st upper mespil rd bank,53.333770,-6.244510,Pedestrians
4,capel st mary street,53.348530,-6.268800,Pedestrians
5,castleknock totem,53.370000,-6.349436,Cyclists
6,charleville mall,53.356950,-6.245480,Cyclists
7,clonskeagh totem,53.306376,-6.237057,Cyclists
8,clontarf james larkin rd,53.368540,-6.170320,Cyclists
9,clontarf pebble beach carpark,53.358250,-6.191070,Cyclists


In [ ]:
def map_user_type(x):
    if "Pedestrians" in x and "Cyclists" in x:
        return "FTF_CYC"
    elif "Pedestrians" in x:
        return "FTF"
    elif "Cyclists" in x:
        return "CYC"
    else:
        return x

counters["User Type"] = counters["User Type"].apply(map_user_type)

In [ ]:
# Dublin city centre map
dublin_center = [53.34, -6.26]
dmap = folium.Map(location=dublin_center, zoom_start=12, width=1800, height=800)

# Add markers to map
for _, row in counters.iterrows():
    icon_color = (
        "blue" if row["User Type"] == "CYC"
        else "purple" if row["User Type"] == "FTF_CYC"
        else "green"
    )

    popup = folium.Popup(row["Bike Counter Locations"], max_width=200)

    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        popup=popup,
        tooltip=row["User Type"],
        icon=folium.Icon(color=icon_color)
    ).add_to(dmap)

dmap

## **COMPARE TO PREPROCESSED LOCATIONS**

In [ ]:
def load_csv_from_blob(blob_name):
    blob_client = container_client.get_blob_client(blob_name)
    stream = blob_client.download_blob()
    data_bytes = stream.readall()
    data_str = data_bytes.decode("utf-8")
    return pd.read_csv(io.StringIO(data_str))


# Connection configuration
AZURE_STORAGE_ACCOUNT = "researchprojectx24104515"
AZURE_STORAGE_KEY = "bxpexO6i+Hz6n1WiipTn+sTCuLPGMS1BogMERrIrHd16DpQ0GLfQ0R33yrSw4MxsDomq5yNMgw1o+AStlx/MjA=="
connection_string = f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

fs = AzureBlobFileSystem(
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)


# Establish connection to preprocessed dataset
CONTAINER_NAME = "preprocessed"
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

file_name = "preprocessed.csv"
data = load_csv_from_blob(file_name)
print(f"Loaded '{file_name}' with shape: {data.shape}")

Loaded 'preprocessed.csv' with shape: (53424, 56)


In [ ]:
# Extract cycle and footfall targets
cycle_t = [col for col in data.columns if col.startswith("CYC_")]
footfall_t = [col for col in data.columns if col.startswith("FTF_")]
targets = cycle_t + footfall_t

In [ ]:
targets

['CYC_Clontarf James Larkin Rd',
 'CYC_Clontarf Pebble Beach Carpark',
 'CYC_Grove Road Totem',
 'CYC_North Strand Rd N B',
 'CYC_Richmond Street Cyclists 1 Merged',
 'CYC_Richmond Street Cyclists 2 Merged',
 'FTF_Aston Quay Merged',
 'FTF_Bachelors Walk Merged',
 'FTF_Baggot St Upper Mespil Rd Bank',
 'FTF_Baggot St Lower Merged',
 'FTF_Capel St Mary St Merged',
 'FTF_College St Dame St Merged',
 'FTF_Dame St Merged',
 'FTF_Dawson St Merged',
 'FTF_Dolier St Burgh Quay Merged',
 'FTF_Grafton St Monsoon Merged',
 'FTF_Grafton St Nassau St Suffolk St',
 'FTF_Grand Canal St Upper Clanwilliam Place Google Merged',
 'FTF_Henry St Merged',
 'FTF_Mary St Merged',
 'FTF_North Wall Quay Samuel Beckett Bridge East',
 'FTF_North Wall Quay Samuel Beckett Bridge West',
 'FTF_Oconnell St Princes St North Merged',
 'FTF_Oconnell St Pennys Merged',
 'FTF_Phibsborough Rd Enniskerry Road',
 'FTF_Richmond St South Portabello Harbour Inbound',
 'FTF_Richmond St South Portabello Harbour Outbound',
 'FTF_T

In [ ]:
counters = counters.sort_values(by=["User Type", "Bike Counter Locations"]).reset_index(drop=True)
counters

,Bike Counter Locations,Latitude,Longitude,User Type
0,castleknock totem,53.370000,-6.349436,CYC
1,charleville mall,53.356950,-6.245480,CYC
2,clonskeagh totem,53.306376,-6.237057,CYC
3,clontarf james larkin rd,53.368540,-6.170320,CYC
4,clontarf pebble beach carpark,53.358250,-6.191070,CYC
5,coast road totem,53.388442,-6.124308,CYC
6,college green bike loop bank of ireland,53.345400,-6.259220,CYC
7,drumcondra cyclists outbound,53.362430,-6.258460,CYC
8,grange road totem,53.286479,-6.277641,CYC
9,griffith avenue clare rd side,53.376570,-6.253620,CYC


In [ ]:
matches = {
 "CYC_Clontarf James Larkin Rd" : "clontarf james larkin rd	",
 "CYC_Clontarf Pebble Beach Carpark" : "clontarf pebble beach carpark",
 "CYC_Grove Road Totem" : "grove road totem",
 "CYC_North Strand Rd N B": "north strand rd n b counter removed for roadworks",
 "CYC_Richmond Street Cyclists 1 Merged": "richmond street inbound",
 "CYC_Richmond Street Cyclists 2 Merged": "richmond street outbound",
 "FTF_Aston Quay Merged": "aston quay fitzgeralds",
 "FTF_Bachelors Walk Merged": "bachelors walk bachelors way",
 "FTF_Capel St Mary St Merged" : "capel st mary street",
 "FTF_College St Dame St Merged" : "college st westmoreland st",
 "FTF_Dame St Merged" : "dame street londis",
 "FTF_Dolier St Burgh Quay Merged" : "d olier st burgh quay",
 "FTF_Grafton St Monsoon Merged" : "grafton st monsoon",
 "FTF_Grafton St Nassau St Suffolk St" : "grafton street nassau street suffolk street",
 "FTF_Henry St Merged" : "henry street coles lane dunnes	",
 "FTF_Mary St Merged" : "mary st jervis st",
 "FTF_Oconnell St Princes St North Merged" : "o connell st princes st north",
 "FTF_Oconnell St Pennys Merged" : "o connell st pennys",
 "FTF_Talbot St Guineys Merged" : "talbot st guineys",
 "FTF_Talbot St Murrays Pharmacy Merged" : "talbot st murrays pharmacy",
 "FTF_Westmoreland St East Fleet St Merged" : "westmoreland street east fleet street removed to return to supplier",
 "FTF_Westmoreland St West Merged" : "westmoreland street west carrolls",
 "FTF_Baggot St Upper Mespil Rd Bank": "baggot st upper mespil rd bank",
 "FTF_Baggot St Lower Merged": "baggot st lower wilton tce inbound",
 "FTF_Dawson St Merged": "dawson street",
 "FTF_Grand Canal St Upper Clanwilliam Place Google Merged": "grand canal st upp clanwilliam place google",
 "FTF_North Wall Quay Samuel Beckett Bridge East": "north wall quay samuel beckett bridge east",
 "FTF_North Wall Quay Samuel Beckett Bridge West": "north wall quay samuel beckett bridge west",
 "FTF_Phibsborough Rd Enniskerry Road": "phibsborough rd enniskerry road",
 "FTF_Richmond St South Portabello Harbour Inbound": "richmond st south portabello harbour inbound",
 "FTF_Richmond St South Portabello Harbour Outbound": "richmond st south portabello harbour outbound",
}

In [ ]:
# Invert dictionary to look for the value
inverse_matches = {v.strip().lower(): k for k, v in matches.items()}

# Normalise text in column to make match
counters["Bike Counter Locations_norm"] = counters["Bike Counter Locations"].str.strip().str.lower()

# Replace values with reversed dictionary
counters["Bike Counter Locations"] = counters["Bike Counter Locations_norm"].map(inverse_matches)

counters = counters.drop(columns=["Bike Counter Locations_norm"])
counters = counters.dropna(subset=["Bike Counter Locations"])
counters = counters.drop(columns=["User Type"])

In [ ]:
counters.rename(columns={
    "Bike Counter Locations": "Target",
}, inplace=True)

counters

,Target,Latitude,Longitude
3,CYC_Clontarf James Larkin Rd,53.368540,-6.170320
4,CYC_Clontarf Pebble Beach Carpark,53.358250,-6.191070
11,CYC_Grove Road Totem,53.329644,-6.270178
13,CYC_North Strand Rd N B,53.358520,-6.241270
15,CYC_Richmond Street Cyclists 1 Merged,53.332160,-6.264400
16,CYC_Richmond Street Cyclists 2 Merged,53.332160,-6.264400
20,FTF_Aston Quay Merged,53.346680,-6.259880
21,FTF_Bachelors Walk Merged,53.347440,-6.260100
22,FTF_Baggot St Lower Merged,53.334540,-6.245620
23,FTF_Baggot St Upper Mespil Rd Bank,53.333770,-6.244510


## **SAVE TO BLOB**

In [ ]:
def save_to_blob(data, filename):
    try:
        blob_name = f"{CONTAINER_NAME}/{filename}"
        csv_data = data.to_csv(index=False)
        with fs.open(blob_name, 'w') as f:
            f.write(csv_data)
        print(f"Saved to {blob_name}")
    except Exception as e:
        print(f"Error saving data to blob storage: {str(e)}")
        return False

In [ ]:
CONTAINER_NAME = "preprocessed"

save_to_blob(counters, "counters.csv")

Saved to preprocessed/counters.csv
